# Kalman Filter Techniques and Statistical Arbitrage

**Updated with yfinance data** | Adapted from [QuantInsti Blog](https://blog.quantinsti.com/kalman-filter-techniques-statistical-arbitrage-china-futures-market-python/)

Original tickers from shared data (Chinese futures):  
`M, SR, FG, CU, SN, NI, RM, AG, OI, CF, HC, J, I, TA, PB, AU, RB, V, JD, MA, TC, A, ZN, SF, P, C, BU, PP, JM, RU, AL, Y, CS, L`

These exact China futures contracts are not available on yfinance. Closest liquid proxies are used below.


## Import libraries

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from pykalman import KalmanFilter
import ffn
import warnings
warnings.filterwarnings('ignore')

# Better colors
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#44BBA4', '#E94F37', '#393E41']
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=COLORS)
%matplotlib inline


## Read data of all contracts

In [ ]:
# Original tickers (China futures) — not on yfinance
original_tickers = [
    'M', 'SR', 'FG', 'CU', 'SN', 'NI', 'RM', 'AG', 'OI', 'CF',
    'HC', 'J', 'I', 'TA', 'PB', 'AU', 'RB', 'V', 'JD', 'MA',
    'TC', 'A', 'ZN', 'SF', 'P', 'C', 'BU', 'PP', 'JM', 'RU',
    'AL', 'Y', 'CS', 'L'
]
print(f"Original tickers ({len(original_tickers)}): {original_tickers}")

# Closest yfinance futures proxies
proxy_map = {
    'CL': 'CL=F', 'BZ': 'BZ=F', 'GC': 'GC=F', 'SI': 'SI=F',
    'HG': 'HG=F', 'NG': 'NG=F', 'ZC': 'ZC=F', 'ZS': 'ZS=F',
    'KC': 'KC=F', 'CT': 'CT=F', 'ZW': 'ZW=F', 'LE': 'LE=F'
}
tickers = list(proxy_map.values())
print(f"\nDownloading {len(tickers)} yfinance proxies...")
raw = yf.download(tickers, start='2018-01-01', end='2026-11-01', progress=False, auto_adjust=True)['Close']
raw.columns = [k for k, v in proxy_map.items() if v in raw.columns]
data = raw.dropna(how='all').ffill().dropna()
print(f"Shape: {data.shape} | {data.index[0].date()} → {data.index[-1].date()}")
data.tail(3)


## Find cointegrated pairs

In [ ]:
def find_cointegrated_pairs(dataframe, critial_level=0.05):
    n = dataframe.shape[1]
    keys = dataframe.columns
    pairs = []
    for i in range(n):
        for j in range(i+1, n):
            score, pvalue, _ = coint(dataframe.iloc[:, i], dataframe.iloc[:, j])
            if pvalue < critial_level:
                pairs.append([keys[i], keys[j], pvalue])
    return pd.DataFrame(pairs, columns=['Asset1', 'Asset2', 'p-value']).sort_values('p-value')

pairs = find_cointegrated_pairs(data)
print(f"Found {len(pairs)} cointegrated pairs")
pairs


## Create functions

In [ ]:
def hurst(ts):
    lags = range(2, 100)
    tau = [np.sqrt(np.std(np.subtract(ts[lag:], ts[:-lag]))) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return poly[0] * 2.0

def half_life(spread):
    spread_lag = spread.shift(1).bfill()
    spread_ret = spread - spread_lag
    spread_lag2 = sm.add_constant(spread_lag)
    model = sm.OLS(spread_ret, spread_lag2).fit()
    hl = int(round(-np.log(2) / model.params.iloc[1], 0))
    return max(hl, 1)

def KalmanFilterAverage(x):
    kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1],
                      initial_state_mean=0, initial_state_covariance=1,
                      observation_covariance=1, transition_covariance=0.01)
    state_means, _ = kf.filter(x.values)
    return pd.Series(state_means.flatten(), index=x.index)

def KalmanFilterRegression(x, y):
    delta = 1e-3
    trans_cov = delta / (1 - delta) * np.eye(2)
    obs_mat = np.expand_dims(np.vstack([[x], [np.ones(len(x))]]).T, axis=1)
    kf = KalmanFilter(n_dim_obs=1, n_dim_state=2,
                      initial_state_mean=[0, 0],
                      initial_state_covariance=np.ones((2, 2)),
                      transition_matrices=np.eye(2),
                      observation_matrices=obs_mat,
                      observation_covariance=2,
                      transition_covariance=trans_cov)
    state_means, _ = kf.filter(y.values)
    return state_means

def backtest(s1, s2, x, y, entryZscore=2, exitZscore=0):
    df1 = pd.DataFrame({'y': y, 'x': x}).dropna()
    state_means = KalmanFilterRegression(KalmanFilterAverage(df1['x']), KalmanFilterAverage(df1['y']))
    df1['hr'] = -state_means[:, 0]
    df1['spread'] = df1.y + (df1.x * df1.hr)
    halflife = half_life(df1['spread'])
    meanSpread = df1.spread.rolling(window=halflife).mean()
    stdSpread = df1.spread.rolling(window=halflife).std()
    df1['zScore'] = (df1.spread - meanSpread) / stdSpread

    df1['long entry'] = ((df1.zScore < -entryZscore) & (df1.zScore.shift(1) >= -entryZscore))
    df1['long exit'] = ((df1.zScore > -exitZscore) & (df1.zScore.shift(1) <= -exitZscore))
    df1['num units long'] = np.nan
    df1.loc[df1['long entry'], 'num units long'] = 1
    df1.loc[df1['long exit'], 'num units long'] = 0
    df1['num units long'] = df1['num units long'].ffill().fillna(0)

    df1['short entry'] = ((df1.zScore > entryZscore) & (df1.zScore.shift(1) <= entryZscore))
    df1['short exit'] = ((df1.zScore < exitZscore) & (df1.zScore.shift(1) >= exitZscore))
    df1['num units short'] = np.nan
    df1.loc[df1['short entry'], 'num units short'] = -1
    df1.loc[df1['short exit'], 'num units short'] = 0
    df1['num units short'] = df1['num units short'].ffill().fillna(0)

    df1['numUnits'] = df1['num units long'] + df1['num units short']
    df1['spread pct ch'] = df1['spread'].diff() / ((df1['x'].abs() * df1['hr'].abs()) + df1['y'])
    df1['port rets'] = df1['spread pct ch'] * df1['numUnits'].shift(1)
    df1['cum rets'] = (1 + df1['port rets'].fillna(0)).cumprod()

    try:
        sharpe = (df1['port rets'].mean() / df1['port rets'].std()) * np.sqrt(252)
    except:
        sharpe = 0.0
    return df1['cum rets'], sharpe, halflife


## Load data

In [ ]:
# Train / test split (~70/30) — mirrors original in-sample / out-sample
split = int(len(data) * 0.70)
train = data.iloc[:split]
test = data.iloc[split:]
print(f"Train: {train.index[0].date()} → {train.index[-1].date()} ({len(train)} days)")
print(f"Test : {test.index[0].date()} → {test.index[-1].date()} ({len(test)} days)")
train.tail(3)


## Choose pairs for out-sample testing

In [ ]:
# Select pairs with Sharpe > 0.5 and final equity > 1.105 (same thresholds as original)
list_sect = []
ret = pd.DataFrame()
for _, row in pairs.iterrows():
    s1, s2 = row['Asset1'], row['Asset2']
    name = f"{s1}-{s2}"
    try:
        tmp, sharpe, hl = backtest(s1, s2, train[s1], train[s2])
        if sharpe > 0.5 and tmp.iloc[-1] > 1.105:
            ret[name] = tmp
            list_sect.append((s1, s2, sharpe, hl, tmp.iloc[-1]))
    except Exception:
        continue

print(f"Pairs selected: {len(list_sect)}")
pd.DataFrame(list_sect, columns=['s1','s2','Sharpe','HalfLife','Final'])


## In sample backtesting

In [ ]:
# In-sample results
if len(ret) > 0:
    ret.iloc[0] = 1
    ret.index = train.index[:len(ret)]
    ret.tail(3)
else:
    print("No pairs met selection criteria")


In [ ]:
# Plot in-sample cumulative returns (better colors)
if len(ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    ret.plot(ax=ax, linewidth=1.8, colormap='Set2')
    ax.set_title('In-Sample Cumulative Returns', fontsize=14, fontweight='bold')
    ax.set_ylabel('Cumulative Returns')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('insample_returns.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Performance stats (ffn — same as original)
if len(ret) > 0:
    perf = ret.calc_stats()
    perf.display()


In [ ]:
# Drawdowns
if len(ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    ffn.to_drawdown_series(ret).plot(ax=ax, linewidth=1.6, colormap='Set2')
    ax.set_title('In-Sample Drawdowns', fontsize=14, fontweight='bold')
    ax.set_ylabel('Drawdown')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('insample_drawdown.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# In-sample portfolio (equal weight)
if len(ret) > 0:
    port = ret.mean(axis=1)
    fig, ax = plt.subplots(figsize=(12, 5))
    port.plot(ax=ax, color=COLORS[0], linewidth=2.2)
    ax.set_title('In-Sample Portfolio Equity', fontsize=14, fontweight='bold')
    ax.set_ylabel('Cumulative Returns')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('insample_portfolio.png', dpi=150, bbox_inches='tight')
    plt.show()
    perf = port.calc_stats()
    print(perf.stats[['total_return','cagr','daily_sharpe','max_drawdown','calmar']])


In [ ]:
# Portfolio drawdown
if len(ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    ffn.to_drawdown_series(port).plot(ax=ax, color=COLORS[3], linewidth=2)
    ax.set_title('In-Sample Portfolio Drawdown', fontsize=14, fontweight='bold')
    ax.set_ylabel('Drawdown')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Out Sample Backtesting

In [ ]:
# Out-sample testing on selected pairs
test_ret = pd.DataFrame()
for s1, s2, *_ in list_sect:
    name = f"{s1}-{s2}"
    try:
        cum, sharpe, hl = backtest(s1, s2, test[s1], test[s2])
        test_ret[name] = cum
    except Exception:
        continue

if len(test_ret) > 0:
    test_ret.iloc[0] = 1
    test_ret.index = test.index[:len(test_ret)]
    print(f"Out-sample pairs: {test_ret.shape[1]}")
    test_ret.tail(3)
else:
    print("No out-sample results")


In [ ]:
# Out-sample cumulative returns
if len(test_ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    test_ret.plot(ax=ax, linewidth=1.8, colormap='Set2')
    ax.set_title('Out-Sample Cumulative Returns', fontsize=14, fontweight='bold')
    ax.set_ylabel('Cumulative Returns')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('outsample_returns.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Out-sample portfolio
if len(test_ret) > 0:
    port_out = test_ret.mean(axis=1)
    fig, ax = plt.subplots(figsize=(12, 5))
    port_out.plot(ax=ax, color=COLORS[1], linewidth=2.2)
    ax.set_title('Out-Sample Portfolio Equity', fontsize=14, fontweight='bold')
    ax.set_ylabel('Cumulative Returns')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('outsample_portfolio.png', dpi=150, bbox_inches='tight')
    plt.show()
    perf = port_out.calc_stats()
    print(perf.stats[['total_return','cagr','daily_sharpe','max_drawdown','calmar']])


In [ ]:
# Out-sample portfolio drawdown
if len(test_ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    ffn.to_drawdown_series(port_out).plot(ax=ax, color=COLORS[3], linewidth=2)
    ax.set_title('Out-Sample Portfolio Drawdown', fontsize=14, fontweight='bold')
    ax.set_ylabel('Drawdown')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('outsample_drawdown.png', dpi=150, bbox_inches='tight')
    plt.show()
